# Graphistry SSO Runbook for Databricks

Interactive diagnostic and login runbook for Graphistry SSO in Databricks notebooks.

## Instructions

1. **Cell 1** — Install dependencies (runs `%pip` and restarts Python)
2. **Cell 2** — Set your server/org configuration
3. **Cell 3** — Run SSO diagnostics (no browser needed)
4. **Cell 4** — Read instructions, then run Cell 5
5. **Cell 5** — Start SSO login (click the link that appears)
6. **Cell 6** — After completing SSO in the browser, run this to get your token
7. **Cell 7** — Verify the token works
8. **Cell 8** — Smoke test: create and plot a graph
9. **Cell 9** — Token refresh test (validates the P0 re-auth bug scenario)
10. **Cell 10** — Summary report

In [ ]:
# Cell 1 — Install dependencies
%pip install graphistry requests
dbutils.library.restartPython()

In [ ]:
# Cell 2 — Configuration
# Edit these values for your environment

SERVER = "graphistry-dev.grph.xyz"  # Graphistry server hostname
PROTOCOL = "https"                  # "https" or "http"
ORG_NAME = None                     # Organization name, or None for site-wide SSO
IDP_NAME = None                     # IdP name, or None for default
SSO_TIMEOUT = None                  # None = non-blocking (recommended for Databricks)

In [ ]:
# Cell 3 — Run SSO Diagnostics
# This cell runs client-side checks against your Graphistry server.
# No browser or SSO login required.

import json
import time
from typing import Any, Dict, List, Optional
from urllib.parse import parse_qs, urlparse
import requests
from IPython.display import display, HTML

DIAGNOSE_SSO_VERSION = "1.0.0"

def _build_base_url(server, protocol="https"):
    return f"{protocol}://{server}"

def _build_sso_login_url(base_url, org_name=None, idp_name=None):
    if org_name is None and idp_name is None:
        return f"{base_url}/api/v2/g/sso/oidc/login/"
    elif org_name is not None and idp_name is None:
        return f"{base_url}/api/v2/o/{org_name}/sso/oidc/login/"
    elif org_name is not None and idp_name is not None:
        return f"{base_url}/api/v2/o/{org_name}/sso/oidc/login/{idp_name}/"
    else:
        return f"{base_url}/api/v2/g/sso/oidc/login/"

def _build_token_poll_url(base_url, state):
    return f"{base_url}/api/v2/o/sso/oidc/jwt/{state}/"

# Diagnostic checks
diag_results = []
diag_hints = []
base_url = _build_base_url(SERVER, PROTOCOL)

def _add(name, passed, detail="", ms=None, data=None):
    diag_results.append({"name": name, "passed": passed, "detail": detail, "ms": ms, "data": data or {}})

def _hint(level, msg):
    diag_hints.append({"level": level, "message": msg})

# Check 1: Server reachable
try:
    t0 = time.time()
    resp = requests.get(base_url, timeout=15, allow_redirects=True)
    ms = (time.time() - t0) * 1000
    _add("Server reachable", resp.status_code < 400, f"HTTP {resp.status_code}", ms)
except Exception as e:
    _add("Server reachable", False, str(e))

auth_url = ""
state = ""

if diag_results[0]["passed"]:
    # Check 2: SSO login endpoint
    sso_url = _build_sso_login_url(base_url, ORG_NAME, IDP_NAME)
    try:
        t0 = time.time()
        resp = requests.post(sso_url, timeout=15)
        ms = (time.time() - t0) * 1000
        body = resp.json()
        status = (body.get("status") or body.get("Status", "")).upper()
        data = body.get("data", {})
        state = data.get("state", "")
        auth_url = data.get("auth_url", "")
        if status == "OK" and state and auth_url:
            _add("SSO login endpoint", True, f"state={state[:12]}...", ms, {"state": state, "auth_url": auth_url})
        else:
            error_msg = body.get("message") or body.get("error") or json.dumps(body)
            _add("SSO login endpoint", False, f"Unexpected: {error_msg}", ms)
    except Exception as e:
        _add("SSO login endpoint", False, str(e))

if auth_url:
    parsed = urlparse(auth_url)
    params = parse_qs(parsed.query)

    # Check 3: PKCE
    challenge = params.get("code_challenge", [None])[0]
    method = params.get("code_challenge_method", [None])[0]
    _add("Auth URL has PKCE", bool(challenge and method == "S256"),
         f"method={method}, challenge={challenge[:12]}..." if challenge else "No code_challenge")

    # Check 4: client_id
    cid = params.get("client_id", [None])[0]
    _add("Auth URL client_id", bool(cid), f"client_id={cid[:12]}..." if cid else "Missing")

    # Check 5: redirect_uri
    redir = params.get("redirect_uri", [None])[0]
    if redir:
        pr = urlparse(redir)
        ok = SERVER in pr.netloc and pr.path.rstrip("/").endswith("/login/callback")
        _add("Auth URL redirect_uri", ok, redir)
    else:
        _add("Auth URL redirect_uri", False, "Missing")

    # Check 6: scopes
    scope_str = params.get("scope", [""])[0]
    scopes = set(scope_str.split())
    missing = {"openid", "profile", "email"} - scopes
    _add("Auth URL scopes", not missing, f"scope={scope_str}" + (f" (missing: {missing})" if missing else ""))

    # Check 7: response_type
    rt = params.get("response_type", [None])[0]
    _add("Auth URL response_type", rt == "code", f"response_type={rt}")

    # Check 8: IdP reachable
    idp_domain = f"{parsed.scheme}://{parsed.netloc}"
    try:
        t0 = time.time()
        resp = requests.head(idp_domain, timeout=10, allow_redirects=True)
        ms = (time.time() - t0) * 1000
        _add("IdP domain reachable", True, f"{parsed.netloc} -> HTTP {resp.status_code}", ms)
    except Exception as e:
        _add("IdP domain reachable", False, f"{parsed.netloc} -> {e}")

    # Check 9: Token poll
    if state:
        poll_url = _build_token_poll_url(base_url, state)
        try:
            t0 = time.time()
            resp = requests.get(poll_url, timeout=15)
            ms = (time.time() - t0) * 1000
            _add("Token poll endpoint", True, f"HTTP {resp.status_code} (expected before login)", ms)
        except Exception as e:
            _add("Token poll endpoint", False, str(e))

# Generate hints
pkce_passed = any(r["name"] == "Auth URL has PKCE" and r["passed"] for r in diag_results)
pkce_failed = any(r["name"] == "Auth URL has PKCE" and not r["passed"] for r in diag_results)
redir_failed = any(r["name"] == "Auth URL redirect_uri" and not r["passed"] for r in diag_results)
idp_failed = any(r["name"] == "IdP domain reachable" and not r["passed"] for r in diag_results)
sso_failed = any(r["name"] == "SSO login endpoint" and not r["passed"] for r in diag_results)

if pkce_passed:
    _hint("WARN", "PKCE is active. If 'Code verifier required' occurs after SSO login, the server's Django CACHES backend may be LocMemCache. Switch to Redis/Memcached.")
if pkce_failed:
    _hint("WARN", "Auth URL missing code_challenge. If IdP is Okta SPA, token exchange will fail.")
if redir_failed:
    _hint("WARN", "redirect_uri mismatch. Check reverse proxy configuration.")
if idp_failed:
    _hint("WARN", "IdP domain unreachable. Check DNS and firewall rules.")
if sso_failed:
    _hint("WARN", "SSO endpoint error. SSO may not be configured on this server.")
_hint("INFO", "For Databricks: use sso_timeout=None (non-blocking). Call graphistry.sso_get_token() in a separate cell after login.")

# Render HTML report
passed = sum(1 for r in diag_results if r["passed"])
failed = sum(1 for r in diag_results if not r["passed"])

html = f"<h3>SSO Diagnostic Report — {PROTOCOL}://{SERVER}</h3>"
if ORG_NAME:
    html += f"<p>Org: <code>{ORG_NAME}</code></p>"
html += '<table style="border-collapse:collapse;">'
html += '<tr style="background:#eee;"><th style="padding:4px 8px;">Status</th><th style="padding:4px 8px;">Check</th><th style="padding:4px 8px;">Detail</th></tr>'
for r in diag_results:
    icon = '&#9989;' if r['passed'] else '&#10060;'
    timing = f" ({r['ms']:.0f}ms)" if r.get('ms') else ""
    html += f'<tr><td style="padding:4px 8px;text-align:center;">{icon}</td>'
    html += f'<td style="padding:4px 8px;">{r["name"]}</td>'
    html += f'<td style="padding:4px 8px;"><code>{r["detail"]}{timing}</code></td></tr>'
html += '</table>'
html += f'<p><strong>{passed} passed, {failed} failed</strong></p>'

if diag_hints:
    html += '<h4>Server-Side Hints</h4><ul>'
    for h in diag_hints:
        color = '#b8860b' if h['level'] == 'WARN' else '#555'
        html += f'<li style="color:{color};"><strong>[{h["level"]}]</strong> {h["message"]}</li>'
    html += '</ul>'

display(HTML(html))

## SSO Login

**Steps:**
1. Run **Cell 5** below — an SSO login link will appear
2. **Click the link** to open your identity provider login in a new tab
3. Complete authentication in the browser
4. Close the browser tab and return here
5. Run **Cell 6** to retrieve your token

> **Important:** We use `sso_timeout=None` (non-blocking mode) to avoid the polling race
> condition that causes "State is invalid" errors. You must manually run Cell 6 after login.

In [ ]:
# Cell 5 — Start SSO Login (non-blocking)
import graphistry

graphistry.register(
    api=3,
    protocol=PROTOCOL,
    server=SERVER,
    is_sso_login=True,
    org_name=ORG_NAME,
    idp_name=IDP_NAME,
    sso_timeout=SSO_TIMEOUT,
    sso_opt_into_type="display",
)

In [ ]:
# Cell 6 — Get Token (run after completing SSO login in browser)
token = graphistry.sso_get_token()

if token:
    display(HTML(f'<p>&#9989; <strong>Token retrieved successfully</strong></p>'))
    print(f"Token prefix: {token[:20]}...")
else:
    display(HTML('<p>&#10060; <strong>No token received.</strong> Did you complete SSO login? Try again.</p>'))
    print("If you haven't logged in yet, click the SSO link in Cell 5, complete login, then re-run this cell.")

In [ ]:
# Cell 7 — Verify token with server
try:
    is_valid = graphistry.verify_token()
    if is_valid:
        display(HTML('<p>&#9989; <strong>Token is valid</strong></p>'))
    else:
        display(HTML('<p>&#10060; <strong>Token verification failed</strong></p>'))
except Exception as e:
    display(HTML(f'<p>&#10060; <strong>Token verification error:</strong> {e}</p>'))

In [ ]:
# Cell 8 — Smoke test: create and plot a graph
import pandas as pd
import numpy as np

nodes = pd.DataFrame({
    "ID": range(10),
    "Name": ["alice", "bob", "carol", "dave", "eve", "frank", "grace", "heidi", "ivan", "judy"],
    "score": np.random.randn(10),
})

edges = pd.DataFrame({
    "src": np.random.choice(10, 20),
    "dst": np.random.choice(10, 20),
})

g = graphistry.nodes(nodes, "ID").edges(edges, "src", "dst")
url = g.plot(render=False)

if url:
    display(HTML(f'<p>&#9989; <strong>Graph created:</strong> <a href="{url}" target="_blank">{url}</a></p>'))
else:
    display(HTML('<p>&#10060; <strong>plot() returned no URL</strong></p>'))

In [ ]:
# Cell 9 — Token refresh test
# This validates the P0 bug scenario: can the token be refreshed
# without hitting "Code verifier required"?

try:
    # First verify current token
    is_valid = graphistry.verify_token()
    if not is_valid:
        display(HTML('<p>&#9888; Token expired, attempting refresh...</p>'))

    # Attempt refresh
    graphistry.refresh()
    refreshed_valid = graphistry.verify_token()

    if refreshed_valid:
        display(HTML('<p>&#9989; <strong>Token refresh succeeded</strong></p>'))
    else:
        display(HTML(
            '<p>&#10060; <strong>Token refresh returned invalid token.</strong></p>'
            '<p>This may indicate the server-side PKCE cache issue (LocMemCache). '
            'Check that Django CACHES is set to Redis or Memcached.</p>'
        ))
except Exception as e:
    err = str(e)
    display(HTML(f'<p>&#10060; <strong>Token refresh failed:</strong> {err}</p>'))
    if "verifier" in err.lower() or "Code verifier" in err:
        display(HTML(
            '<p style="color:red;"><strong>P0 BUG CONFIRMED:</strong> '
            '"Code verifier required" during refresh. '
            'Server CACHES backend must be changed from LocMemCache to Redis/Memcached.</p>'
        ))
    elif "sso" in err.lower() or "login" in err.lower():
        display(HTML(
            '<p>Refresh triggered re-authentication. '
            'You may need to re-run Cell 5 to start a new SSO login.</p>'
        ))

In [ ]:
# Cell 10 — Summary Report

summary_checks = []

# Diagnostic results from Cell 3
for r in diag_results:
    summary_checks.append((r["name"], r["passed"], r["detail"]))

# Token status
try:
    tok = graphistry.api_token()
    summary_checks.append(("Token acquired", bool(tok), f"prefix={tok[:12]}..." if tok else "No token"))
except Exception:
    summary_checks.append(("Token acquired", False, "Error checking token"))

# Token validity
try:
    valid = graphistry.verify_token()
    summary_checks.append(("Token valid", valid, ""))
except Exception as e:
    summary_checks.append(("Token valid", False, str(e)))

# Build HTML summary
html = f'<h3>SSO Runbook Summary — {PROTOCOL}://{SERVER}</h3>'
html += '<table style="border-collapse:collapse;">'
html += '<tr style="background:#eee;"><th style="padding:4px 8px;">Status</th><th style="padding:4px 8px;">Check</th><th style="padding:4px 8px;">Detail</th></tr>'
passed_total = 0
failed_total = 0
for name, ok, detail in summary_checks:
    icon = '&#9989;' if ok else '&#10060;'
    if ok:
        passed_total += 1
    else:
        failed_total += 1
    html += f'<tr><td style="padding:4px 8px;text-align:center;">{icon}</td>'
    html += f'<td style="padding:4px 8px;">{name}</td>'
    html += f'<td style="padding:4px 8px;"><code>{detail}</code></td></tr>'
html += '</table>'
html += f'<p><strong>{passed_total} passed, {failed_total} failed out of {passed_total + failed_total}</strong></p>'

if failed_total == 0:
    html += '<p style="color:green;"><strong>All checks passed. SSO is working correctly.</strong></p>'
else:
    html += '<p style="color:red;"><strong>Some checks failed. Review the diagnostic hints in Cell 3.</strong></p>'

display(HTML(html))